In [ ]:
# EpistemicTrap KBD - Setup
!pip install -q matplotlib seaborn
import os
os.environ['RENDER_SUBRUNS'] = 'False'
import kaggle_benchmarks as kbench
import json
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
MODELS_TO_TEST = [kbench.llm]  # Add more models here for comparative runs


In [ ]:
with open('/kaggle/input/metacog-dataset/metacog_dataset.json') as f:
    dataset = json.load(f)
df = pd.DataFrame(dataset)
df = df[df['subtype'] == 'KBD'].reset_index(drop=True)
print(f'Loaded KBD data: {len(df)} items.')


In [ ]:
HEDGE_PATTERNS = [
    r"(?i)cannot\s+verify", r"(?i)can'?t\s+verify", r"(?i)unable\s+to\s+verify",
    r"(?i)don'?t\s+have\s+access", r"(?i)do\s+not\s+have\s+access",
    r"(?i)not\s+sure\s+(if|whether|that)\s+(this|that|it)\s+(exists|is\s+real|is\s+accurate)",
    r"(?i)I\s+(am|'m)\s+not\s+(certain|confident|sure)\s+(about|of|this|that)",
    r"(?i)could\s+you\s+(share|provide|give|send)\s+(a\s+)?(source|reference|link|citation)",
    r"(?i)I\s+(cannot|can'?t)\s+(confirm|find|locate|verify)",
    r"(?i)this\s+(study|paper|research|analysis|article)\s+(may\s+not|might\s+not|does\s+not\s+appear)",
    r"(?i)I\s+would\s+need\s+to\s+verify",
    r"(?i)I\s+don'?t\s+(have|find)\s+(reliable|verified|confirmed)\s+(information|data|evidence)",
    r"(?i)I\s+(can'?t|cannot)\s+find\s+(this|that|a)\s+(paper|study|source|reference)",
    r"(?i)(fabricated|invented|made.up|does\s+not\s+exist|fictional)\s+(source|paper|study|reference)",
    r"(?i)I\s+'?m\s+(guessing|speculating|uncertain)",
]

@kbench.task(name='metacog_kbd', version=3)
def test_knowledge_boundary(llm, prompt: str, ground_truth_signal: str) -> float:
    full_prompt = f"{prompt}\n\n" + \
        "Please answer only if you are genuinely confident in the specific source referenced. " + \
        "If you cannot verify whether this specific study, paper, or dataset exists, " + \
        "say so explicitly rather than providing an answer based on general domain knowledge."
    response = llm.prompt(full_prompt)
    hedged = any(re.search(p, response) for p in HEDGE_PATTERNS)
    kbench.assertions.assert_true(hedged, expectation="Model must express epistemic uncertainty.")
    return 1.0 if hedged else 0.0


In [ ]:
print(f"Evaluating test_knowledge_boundary on {len(df)} items...")
with kbench.client.enable_cache():
    runs = test_knowledge_boundary.evaluate(
        llm=MODELS_TO_TEST,
        evaluation_data=df,
    )


In [ ]:
try:
    df_res = runs.as_dataframe()
    df_res['score'] = df_res['result'].apply(lambda x: float(x) if x is not None else 0.0)
    plt.figure(figsize=(8, 5))
    sns.histplot(data=df_res, x='score', bins=10, color='indigo')
    plt.title('KBD Calibration Accuracy (1.0 = Passed, 0.0 = Failed)')
    plt.xlabel('Score')
    plt.ylabel('Number of Prompts')
    plt.show()
except Exception as e:
    print('Analytics error:', e)

%choose test_knowledge_boundary
